# CE432P: Hydraulic Engineering
## Unit 5 — Computational Fluid Dynamics & Hydrodynamic Modelling (Day 43)
**Topic:** 1D Hydrodynamic Wave Routing, Discretisation & HEC-RAS 2D Comparison

---
### 1. Shallow Water Equations (SWE)
The 1D Saint-Venant continuity and momentum equations describe unsteady open channel flow:
$$\frac{\partial A}{\partial t} + \frac{\partial Q}{\partial x} = 0$$
$$\frac{\partial Q}{\partial t} + \frac{\partial}{\partial x}\left(\frac{Q^2}{A}\right) + g A \frac{\partial y}{\partial x} + g A (S_f - S_0) = 0$$

In HEC-RAS 2D, the subgrid bathymetry approach solves either the Diffusion Wave approximation or the full Shallow Water Equations on an unstructured flexible mesh.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1D Kinematic Wave Flood Routing Simulation (Explicit Upwind Scheme)
L = 10000.0   # 10 km channel reach
dx = 200.0    # grid spacing 200 m
nx = int(L / dx) + 1
x = np.linspace(0, L, nx)

dt = 10.0     # time step (CFL check: c * dt / dx <= 1)
total_time = 7200.0 # 2 hours
nt = int(total_time / dt)

# Inflow flood hydrograph at x = 0
time_series = np.linspace(0, total_time, nt)
Q_base = 20.0
Q_peak = 120.0
t_peak = 1800.0 # 30 mins
inflow_Q = Q_base + (Q_peak - Q_base) * np.exp(-((time_series - t_peak) / 900.0)**2)

# Routing setup
Q_grid = np.ones((nt, nx)) * Q_base
c_celerity = 2.5 # kinematic wave celerity c = dQ/dA

for t in range(0, nt - 1):
    Q_grid[t, 0] = inflow_Q[t]
    for i in range(1, nx):
        Q_grid[t + 1, i] = Q_grid[t, i] - (c_celerity * dt / dx) * (Q_grid[t, i] - Q_grid[t, i - 1])

plt.figure(figsize=(9, 5.5))
plt.plot(time_series / 60.0, Q_grid[:, 0], label="Inflow Hydrograph (x = 0 km)", lw=2.5, color="navy")
plt.plot(time_series / 60.0, Q_grid[:, int(nx * 0.5)], label=f"Mid-Reach Hydrograph (x = {L*0.5/1000:.1f} km)", lw=2, color="orange")
plt.plot(time_series / 60.0, Q_grid[:, -1], label=f"Outflow Hydrograph (x = {L/1000:.1f} km)", lw=2, color="green")

plt.xlabel("Time (minutes)", fontsize=12)
plt.ylabel("Discharge $Q$ (m³/s)", fontsize=12)
plt.title("1D Hydrodynamic Flood Wave Routing Along Channel", fontsize=14, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.show()
